In [5]:
import mlflow
from mlflow.tracking import MlflowClient

experiment_name = "linreg-vs-randforest"

mlflow.set_tracking_uri("http://localhost:5001")
mlflow.set_experiment(experiment_name)

client = MlflowClient()

print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Active Experiment: {mlflow.get_experiment_by_name(experiment_name)}")
    
with mlflow.start_run(run_name="connection-check"):
    mlflow.log_param("test_param", "test_value")
    print("✓ Successfully connected to MLflow!")

2026/03/06 14:16:36 INFO mlflow.tracking.fluent: Experiment with name 'linreg-vs-randforest' does not exist. Creating a new experiment.


MLflow Tracking URI: http://localhost:5001
Active Experiment: <Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1772795796274, experiment_id='1', last_update_time=1772795796274, lifecycle_stage='active', name='linreg-vs-randforest', tags={}>
✓ Successfully connected to MLflow!
🏃 View run connection-check at: http://localhost:5001/#/experiments/1/runs/3f44365593c245708d003a81cc308def
🧪 View experiment at: http://localhost:5001/#/experiments/1


In [6]:
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
import pandas as pd


train_path = Path("../data/processed/train.csv")
test_path = Path("../data/processed/test.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

target_col = "Class"
feature_cols = [c for c in train_df.columns if c != target_col]

X_train = train_df[feature_cols]
y_train = train_df[target_col].astype(int)
X_test = test_df[feature_cols]
y_test = test_df[target_col].astype(int)

experiments = [
    {
        "model_name": "LogisticRegression",
        "estimator": LogisticRegression,
        "params": {
            "solver": "lbfgs",
            "max_iter": 1500,
            "class_weight": "balanced",
            "random_state": 42,
        },
    },
    {
        "model_name": "RandomForest",
        "estimator": RandomForestClassifier,
        "params": {
            "n_estimators": 300,
            "max_depth": 12,
            "min_samples_split": 2,
            "class_weight": "balanced_subsample",
            "random_state": 42,
            "n_jobs": -1,
        },
    },
]

rows = []
for exp in experiments:
    model_name = exp["model_name"]
    params = exp["params"]
    model = exp["estimator"](**params)

    with mlflow.start_run(run_name=f"fraud-{model_name}"):
        mlflow.set_tag("dataset", "credit-card-fraud")
        mlflow.set_tag("model_family", model_name)
        mlflow.log_param("target", target_col)
        mlflow.log_param("train_rows", len(train_df))
        mlflow.log_param("test_rows", len(test_df))
        mlflow.log_params(params)

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_score = model.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision_fraud": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
            "recall_fraud": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
            "f1_fraud": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
            "pr_auc": average_precision_score(y_test, y_score),
            "roc_auc": roc_auc_score(y_test, y_score),
        }

        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, name="model")

        run_id = mlflow.active_run().info.run_id
        rows.append({"model": model_name, "run_id": run_id, **metrics})

pd.DataFrame(rows).sort_values("recall_fraud", ascending=False)

/Users/stepa/Study/master/mlops/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/stepa/Study/master/mlops/.venv/lib/python3.11/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
 

🏃 View run fraud-LogisticRegression at: http://localhost:5001/#/experiments/1/runs/ff35ed11bc8741e080c4b46a03064af9
🧪 View experiment at: http://localhost:5001/#/experiments/1


/Users/stepa/Study/master/mlops/.venv/lib/python3.11/site-packages/mlflow/models/model.py:1209: FutureWarning: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.
  flavor.save_model(path=local_path, mlflow_model=mlflow_model, **kwargs)


🏃 View run fraud-RandomForest at: http://localhost:5001/#/experiments/1/runs/9623fdf9fc484fdcb201b1c76b916c51
🧪 View experiment at: http://localhost:5001/#/experiments/1


,model,run_id,accuracy,precision_fraud,recall_fraud,f1_fraud,pr_auc,roc_auc
0,LogisticRegression,ff35ed11bc8741e080c4b46a03064af9,0.973543,0.054959,0.887755,0.103510,0.706704,0.975932
1,RandomForest,9623fdf9fc484fdcb201b1c76b916c51,0.999421,0.842105,0.816327,0.829016,0.816397,0.979533
